# Von Mises navigation policy

A compact, self-contained illustration of a **von Mises** steering policy. An agent at `x_start` wants to reach a target `g`; each step it samples a heading from a von Mises distribution

```
theta | x  ~  V( mu = heading-to-target,  kappa )  ,
step  =  step_len * ( cos theta, sin theta )
```

so that the *mean* heading is always directed at the goal, and `kappa` controls how much the actual step deviates from that ideal direction (kappa = 0 is a uniform, unbiased-but-random walk; larger kappa is tighter and more goal-directed).

The figure has two rows, one column per `kappa` in `{0, 0.5, 2, 5}`:

- **Top row** — the exact von Mises density `V(theta | mu, kappa)` rendered as a radial modulation around the policy radius, with the goal direction marked.
- **Bottom row** — 10 independent trajectories produced by the stepwise von Mises sampling, starting from the fixed agent origin (black dot) toward the target (triangle).

Why the von Mises distribution? Under discrete steps of constant length and no other constraints, the heading policy that maximizes expected progress toward the goal is exactly a von Mises distribution aimed at the target: its mean is the inferred target direction, and its concentration quantifies how strongly the agent prioritizes moving closer to the goal over heading noise - so fitting such models to observed headings estimates behavioral policies directly.

This is a *policy* demo — it characterizes the stochastic heading distribution a controller would use — and is the visual companion to the *likelihood* use of the von Mises distribution in `notebooks_r/beetles.ipynb` and `notebooks_r/zebra_escape.ipynb`.


## Setup

Select the R (`ir`) kernel and follow the [R setup](../getting-started-r.md)
page for the compiler and kernel prerequisites. Run the cells in order.
Notebook execution is local by design (tests/test_notebooks.py).


In [ ]:
required <- c("circular", "ggplot2", "ragg", "IRdisplay", "patchwork")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}


In [ ]:
versions <- c(R = as.character(getRversion()),
  vapply(c("circular", "ggplot2", "ragg", "IRdisplay", "patchwork"),
    function(p) as.character(packageVersion(p)), character(1)))
versions


In [ ]:
suppressPackageStartupMessages({
  library(circular)
  library(ggplot2)
  library(patchwork)
})


In [ ]:
repo_root <- if (file.exists(file.path("..", "R", "unwrap_family.R"))) {
  ".."
} else if (file.exists(file.path("R", "unwrap_family.R"))) {
  "."
} else {
  stop("Run from notebooks_r/ or the repository root.")
}


In [ ]:
FIG_DIR <- file.path(repo_root, "figures", "von_mises_policy")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)
# Render headless PNGs with ragg; keep archival vector PDFs alongside them.

show_figure <- function(plot, name, width, height, res = 110) {
  file <- file.path(FIG_DIR, paste0(name, ".png"))
  ragg::agg_png(file, width = width, height = height, units = "in",
                res = res)
  print(plot)
  invisible(dev.off())
  IRdisplay::display_png(file = file)
  invisible(NULL)
}

theme_set(theme_minimal(base_size = 12))
save_figure <- function(plot, name, width = 8, height = 5) {
  ggsave(file.path(FIG_DIR, paste0(name, ".pdf")), plot, width = width, height = height)
  show_figure(plot, name, width, height)
}


In [ ]:
SEED <- 12345
quick_run <- Sys.getenv("UNWRAP_R_QUICK") == "1"
CHAINS <- if (quick_run) 2L else 4L
ITER <- if (quick_run) 3000L else 4000L
WARMUP <- if (quick_run) 1500L else 2000L
# These are local execution budgets, not convergence claims. Hard diagnostic
# gates below apply to both budgets; increase sampling if they fail.


In [ ]:
wrap <- function(x) atan2(sin(x), cos(x))
circ_mean <- function(x) atan2(mean(sin(x)), mean(cos(x)))
resultant <- function(x) Mod(mean(exp(1i * x)))
softplus_stable <- function(x) pmax(x, 0) + log1p(exp(-abs(x)))
log1p_exp <- softplus_stable
A1_safe <- function(k) {
  z <- besselI(k, 1, expon.scaled = TRUE) / besselI(k, 0, expon.scaled = TRUE)
  if (!is.null(dim(k))) dim(z) <- dim(k)
  z
}
vm_logpdf <- function(y, mu, kappa) {
  kappa * (cos(y - mu) - 1) - log(2 * pi) - log(besselI(kappa, 0, expon.scaled = TRUE))
}
hdi <- function(x, prob = 0.95) {
  x <- sort(as.numeric(x))
  stopifnot(length(x) > 1L, all(is.finite(x)))
  gap <- min(length(x) - 1L, floor(prob * length(x)))
  ii <- seq_len(length(x) - gap)
  j <- ii[which.min(x[ii + gap] - x[ii])]
  c(low = x[j], high = x[j + gap])
}
interval <- function(x, angular = FALSE, prob = 0.95) {
  m <- if (angular) circ_mean(x) else mean(x)
  z <- if (angular) m + wrap(x - m) else x
  c(mean = m, hdi(z, prob))
}
vmu <- function(m) circular::circular(
    m, units = "radians", template = "none", modulo = "2pi")
vm_ctrl <- list(units = "radians", template = "none", modulo = "2pi")
draw_vm <- function(mu, kappa) {
  wrap(as.numeric(circular::rvonmises(1, mu = vmu(mu), kappa = kappa,
                                     control.circular = vm_ctrl)))
}


## Figure style

Figures use portable sans text and the shared minimal theme. Each figure
renders inline through the headless ragg PNG helper and saves a vector PDF
under `figures/von_mises_policy/`. The generated directory is gitignored.


## 1. The steering policy

The policy and its configuration, kept inline: `goal_heading` aims the mean von Mises direction at the target `g` from the current position, `vonmises_pdf` is the exact density `V(theta | mu, kappa)`, and the config fixes the origin, target, the concentration levels in `kappas`, the step size and the (per-column) sampling seeds, so every panel is comparable.


In [ ]:
goal_heading <- function(x, y) atan2(y[2] - x[2], y[1] - x[1])
vonmises_pdf <- function(theta, mu, kappa) exp(vm_logpdf(theta, mu, kappa))
x_start <- c(0.0, 0.0)
target <- c(0.0, 10.0)
kappas <- c(0, 0.5, 2, 5)
T_steps <- 300L
step_len <- 0.15
num_traj <- 10L
rng_seed <- 12345L
R_policy <- 1.75
amp <- 1.0
# Keep all geometry and trajectory counts fixed even in quick mode.
# EQUIVALENCE NOTE: circular exports rvonmises, not rmv. The scalar
# rvonmises call uses the tutorial's warning-free control.circular pattern.


## 2. Evaluate the densities and sample trajectories

The top-row radial profile evaluates the exact von Mises density on a fine angle grid for each `kappa` (rescaled to a small radial modulation around the policy radius). The bottom-row trajectories are the Monte-Carlo stepwise sampling of the same policy - the two rows are two views of the *same* distribution family, and their agreement is the point of the figure.


In [ ]:
theta_grid <- (0:1023) * (2 * pi / 1024)
mu0 <- goal_heading(x_start, target)
radials <- lapply(kappas, function(kappa) {
  pdf <- vonmises_pdf(theta_grid, mu0, kappa)
  R_policy + amp * (pdf - min(pdf)) / (max(pdf) - min(pdf) + 1e-12)
})
trajectories <- vector("list", length(kappas))
for (i in seq_along(kappas)) {
  set.seed(rng_seed + i - 1L)
  pos <- array(0, dim = c(T_steps + 1L, num_traj, 2L))
  for (j in seq_len(num_traj)) pos[1, j, ] <- x_start
  for (t in seq_len(T_steps)) {
    for (j in seq_len(num_traj)) {
      mu <- goal_heading(pos[t, j, ], target)
      theta <- draw_vm(mu, kappas[i])
      pos[t + 1L, j, ] <- pos[t, j, ] + step_len * c(cos(theta), sin(theta))
    }
  }
  steps <- sqrt(diff(pos[, 1, 1])^2 + diff(pos[, 1, 2])^2)
  stopifnot(all(abs(steps - step_len) < 1e-10))
  trajectories[[i]] <- pos
}
# R and NumPy use different von Mises random-number implementations and
# precision (R doubles versus Python float32); paths have distributional,
# not bitwise, parity. Geometry, seeds, counts, and step lengths match.


## 3. Draw the figure

The following cells draw the two-row figure and saves it (300 dpi vector) under `figures/von_mises_policy/`; the cell renders it inline at 100 dpi.


In [ ]:
policy_panels <- trajectory_panels <- vector("list", length(kappas))
for (i in seq_along(kappas)) {
  polar <- data.frame(theta = c(theta_grid, 2 * pi),
    radius = c(radials[[i]], radials[[i]][1]),
    reference = R_policy)
  policy_panels[[i]] <- ggplot(polar, aes(theta, radius)) +
    geom_path(aes(group = 1), linewidth = 0.6) +
    geom_path(aes(y = reference, group = 1), linetype = "dotted", linewidth = 0.3) +
    geom_point(data = data.frame(theta = mu0, radius = R_policy + amp + 0.22),
      aes(theta, radius), inherit.aes = FALSE, shape = 17) +
    scale_x_continuous(limits = c(0, 2 * pi), expand = c(0, 0)) +
    scale_y_continuous(limits = c(0, R_policy + amp + 0.3), expand = c(0, 0)) +
    coord_polar(start = pi / 2, direction = -1) +
    labs(title = paste("kappa =", kappas[i]), x = "", y = "Policy density") +
    theme_void(base_size = 12)
  pos <- trajectories[[i]]
  seg <- do.call(rbind, lapply(seq_len(num_traj), function(j) {
    data.frame(iteration = seq_len(T_steps), trajectory = factor(j),
      x = pos[-(T_steps + 1L), j, 1], y = pos[-(T_steps + 1L), j, 2],
      xend = pos[-1L, j, 1], yend = pos[-1L, j, 2])
  }))
  # Segments encode consecutive time points explicitly: no sorting of an
  # x/y path can change a trajectory's temporal order.
  trajectory_panels[[i]] <- ggplot(seg, aes(x, y, colour = trajectory)) +
    geom_segment(aes(xend = xend, yend = yend), linewidth = 0.3, alpha = 0.9) +
    annotate("point", x = x_start[1], y = x_start[2], colour = "black", size = 2) +
    annotate("point", x = target[1], y = target[2], colour = "black", shape = 17, size = 2.5) +
    scale_colour_manual(values = c("#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
      "#8C564B", "#E377C2", "#7F7F7F", "#BCBD22", "#17BECF")) +
    coord_equal(xlim = c(-7.5, 7.5), ylim = c(-3.5, 10.8)) +
    theme_void(base_size = 12) + theme(legend.position = "none")
}


In [ ]:
column_1 <- patchwork::wrap_plots(policy_panels[[1]], trajectory_panels[[1]], ncol = 1)
save_figure(column_1, "vonmises_navigation_kappa_0", 4, 8)


In [ ]:
column_2 <- patchwork::wrap_plots(policy_panels[[2]], trajectory_panels[[2]], ncol = 1)
save_figure(column_2, "vonmises_navigation_kappa_0_5", 4, 8)


In [ ]:
column_3 <- patchwork::wrap_plots(policy_panels[[3]], trajectory_panels[[3]], ncol = 1)
save_figure(column_3, "vonmises_navigation_kappa_2", 4, 8)


In [ ]:
column_4 <- patchwork::wrap_plots(policy_panels[[4]], trajectory_panels[[4]], ncol = 1)
save_figure(column_4, "vonmises_navigation_kappa_5", 4, 8)


In [ ]:
p_navigation <- patchwork::wrap_plots(c(policy_panels, trajectory_panels), ncol = length(kappas))
save_figure(p_navigation, "vonmises_navigation", 12, 6)


## How to read the panels

- **kappa = 0**: the top-row curve is a perfect circle (the density is uniform in theta) and the bottom-row trajectories are a random walk — no net progress on average.
- **kappa = 0.5 / 2 / 5**: the radial curve progressively sharpens in the direction of the goal (a lobe on the goal side), and the trajectories progressively align toward the target. By `kappa = 5` the walk makes steady, low-deviation progress.
- The fixed `step_len`, `T_steps`, and origin keep the panels directly comparable, isolating the single effect of concentration `kappa`.

Note the two rows are drawn from the *same* distribution family: the top row is the analytic PDF; the bottom row is the Monte-Carlo realisation of that PDF integrated over time. Agreement between them is the point.
